# BigQuery Studio Visual Edition · Act 2: Customer 360 Subgraph, `GRAPH_EXPAND` & 3-Method Scorecard

> **Designed for BigQuery Studio (Colab Enterprise in the Google Cloud Console).**
> Combines interactive `%%bigquery --graph` Customer 360 visualisations with declarative `GRAPH_EXPAND` + `AGG()` semantic queries and the 3-method benchmark scorecard.

## 1 · Interactive Customer 360 Subgraph for `Abby Noland` (`PER-1cdfdb9f52ddb25a`)

Run the cell below in BigQuery Studio to render the complete multi-hop Customer 360 subgraph around `PER-1cdfdb9f52ddb25a` (`Abby Noland`):
- **5 `SourceRecord` nodes** (`CRM-5d268dd4`, `LOY-ac677db5`, `SUP-da73f0e1`, `POS-1b222aed`, `ENR-31fd9a08`)
- Their shared **`IdentifierNode`**, **`Email`**, **`Phone`**, **`Account`**, **`Address`**, and **`Household`** nodes.

In [1]:
%%bigquery --graph
GRAPH `all-things-cdp.cdp.cdp_identity_graph`
MATCH (i:IdentifierNode)<-[hi:HAS_IDENTIFIER]-(r:SourceRecord)-[res:RESOLVES_TO]->(p:Person)-[la:LIVES_AT]->(a:Address)
WHERE p.person_id = 'PER-1cdfdb9f52ddb25a'
RETURN i, hi, r, res, p, la, a

Rendered Customer 360 subgraph for PER-1cdfdb9f52ddb25a (Abby Noland: 5 SourceRecords across CRM, LOYALTY, SUPPORT, POS, ENRICH).

## 2 · Declarative Semantic Graph Aggregation (`FROM GRAPH_EXPAND`)

Query `all-things-cdp.cdp.cdp_semantic_graph` using `GRAPH_EXPAND` and `AGG()` to compare cluster counts and hairball membership across all 7 retail brands/sources without join fan-out.

In [2]:
%%bigquery
SELECT
  Profile_sources AS brand,
  AGG(Profile_profile_count)                    AS profiles,
  AGG(Membership_link_count)                    AS links,
  AGG(Identifier_identifier_count)              AS distinct_identifiers,
  AGG(Profile_profiles_touching_hubs)           AS touching_hubs,
  AGG(Profile_baseline_wesid_count)             AS baseline_clusters,
  AGG(Profile_weighted_wesid_count)             AS weighted_clusters,
  AGG(Profile_composable_wesid_count)           AS composable_clusters,
  AGG(Profile_profiles_in_baseline_hairballs)   AS in_baseline_hairballs,
  AGG(Profile_profiles_in_weighted_hairballs)   AS in_weighted_hairballs,
  AGG(Profile_profiles_in_composable_hairballs) AS in_composable_hairballs
FROM GRAPH_EXPAND("all-things-cdp.cdp.cdp_semantic_graph")
GROUP BY brand
ORDER BY profiles DESC

Semantic Graph Measure Rollup by Source Brand (all-things-cdp.cdp.cdp_semantic_graph)
brand    profiles  links  distinct_ids  touching_hubs  baseline_clusters  weighted_clusters  composable_clusters  baseline_hairball  composable_hairball
-------  --------  -----  ------------  -------------  -----------------  -----------------  -------------------  -----------------  -------------------
POS      5593      2190   2117          505            3918               5340               5192                 650                0                  
CRM      4155      8451   7834          381            2002               4125               4125                 1455               0                  
ECOM     3752      5573   5268          355            1986               3716               3719                 1185               0                  
LOYALTY  3248      6674   6235          307            1732               3233               3232                 1087               0               

brand,profiles,links,distinct_ids,touching_hubs,baseline_clusters,weighted_clusters,composable_clusters,baseline_hairball,composable_hairball
POS,5593,2190,2117,505,3918,5340,5192,650,0
CRM,4155,8451,7834,381,2002,4125,4125,1455,0
ECOM,3752,5573,5268,355,1986,3716,3719,1185,0
LOYALTY,3248,6674,6235,307,1732,3233,3232,1087,0
ENRICH,1842,1945,1891,189,1075,1833,1835,526,0
SUPPORT,968,1068,1039,98,662,962,965,224,0
CALL,442,329,321,56,325,442,442,68,0


## 3 · Side-by-Side 3-Method Benchmark (`all-things-cdp.cdp.v_method_comparison`)

In [3]:
%%bigquery
SELECT
  pathology,
  method,
  tp,
  fp,
  fn,
  ROUND(precision, 4) AS precision,
  ROUND(recall, 4)    AS recall,
  ROUND(f1, 4)        AS f1
FROM `all-things-cdp.cdp.v_method_comparison`
WHERE pathology IN ('ALL', 'hub', 'household', 'supersession', 'semantic_fuzzy')
ORDER BY
  CASE pathology
    WHEN 'ALL' THEN 0 WHEN 'hub' THEN 1 WHEN 'household' THEN 2
    WHEN 'supersession' THEN 3 WHEN 'semantic_fuzzy' THEN 4
  END,
  CASE method WHEN 'baseline_cc' THEN 1 WHEN 'weighted_cc' THEN 2 ELSE 3 END

3-Method Benchmark Scorecard by Pathology Class (all-things-cdp.cdp.v_method_comparison)
pathology       method          tp    fp        fn     precision  recall  f1    
--------------  --------------  ----  --------  -----  ---------  ------  ------
ALL             baseline_cc     8064  13488214  10710  0.0006     0.4295  0.0012
ALL             weighted_cc     7566  321       11208  0.9593     0.403   0.5675
ALL             composable_cdp  8274  45        10500  0.9946     0.4407  0.6107
hub             baseline_cc     2423  13488039  3294   0.0002     0.4238  0.0004
hub             weighted_cc     2079  64        3638   0.9701     0.3637  0.529 
hub             composable_cdp  2274  12        3443   0.9948     0.3978  0.5683
household       baseline_cc     151   54        92     0.7366     0.6214  0.6741
household       weighted_cc     147   100       96     0.5951     0.6049  0.6   
household       composable_cdp  153   0         90     1.0        0.6296  0.7727
supersession    base

pathology,method,tp,fp,fn,precision,recall,f1
ALL,baseline_cc,8064,13488214,10710,0.0006,0.4295,0.0012
ALL,weighted_cc,7566,321,11208,0.9593,0.403,0.5675
ALL,composable_cdp,8274,45,10500,0.9946,0.4407,0.6107
hub,baseline_cc,2423,13488039,3294,0.0002,0.4238,0.0004
hub,weighted_cc,2079,64,3638,0.9701,0.3637,0.529
hub,composable_cdp,2274,12,3443,0.9948,0.3978,0.5683
household,baseline_cc,151,54,92,0.7366,0.6214,0.6741
household,weighted_cc,147,100,96,0.5951,0.6049,0.6
household,composable_cdp,153,0,90,1.0,0.6296,0.7727
supersession,baseline_cc,28,29,62,0.4912,0.3111,0.381
